# OpenPlaque standalone TPV + PAV comparison

This notebook reproduces the earlier OpenPlaque plaque-volume definitions and then
computes experimental PAV using a candidate outer-vessel wall.

It reports separately:

- **Strict nnU-Net TPV**: all label-2 plaque voxels.
- **Earlier best-estimate 3-vessel TPV proxy**: strict-core HU classes plus HU-selected
  vessel-context candidates, matching the earlier plaque-type notebook.
- **Earlier best-estimate TPV proxy + left main**: adds the prior clinical left-main
  calcium volume of 54 mm³ for TPV comparison only.
- **Strict-core PAV** and **best-estimate 3-vessel PAV** using the same candidate
  outer-wall denominator.

The 54 mm³ left-main add-on is **not included in PAV** because there is no corresponding
left-main outer-vessel denominator in this prototype.

Research use only. Not clinically validated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys, subprocess
from pathlib import Path

REPO_DIR = Path('/content/OpenPlaque')
if REPO_DIR.exists():
    subprocess.run(['rm', '-rf', str(REPO_DIR)], check=True)

subprocess.run([
    'git', 'clone', '--branch', 'pav-outer-wall-prototype',
    'https://github.com/pazzani/OpenPlaque.git', str(REPO_DIR)
], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))

import openplaque
from openplaque.plaque_volume import (
    estimate_best_estimate_plaque_volume,
    segmentation_mask_with_plaque_proxy,
)
from openplaque.pav import estimate_outer_wall_candidate, compute_pav, show_pav_overlay

print('OpenPlaque:', openplaque.__file__)
print('Standalone TPV/PAV imports OK.')


## Load the UCLA study and saved artery masks

This uses the same `Full_DICOM.zip`, artery-series fallback mapping, and Series 7/reference
spacing fallback used by the earlier analysis.


In [ ]:
import os
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk
import matplotlib.pyplot as plt

from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series

DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = DRIVE_ROOT / 'Full_DICOM.zip'

MASK_DIR_CANDIDATES = [
    DRIVE_ROOT / 'UCLA_Plaque_Type_Estimates/nnunet_masks',
    DRIVE_ROOT / 'UCLA_Plaque_Context_Verification/nnunet_masks',
]
MASK_DIR = next(
    (p for p in MASK_DIR_CANDIDATES
     if all((p / f'{a}.nii.gz').exists() for a in ['LAD', 'LCX', 'RCA'])),
    None,
)

if not STUDY_ZIP.exists():
    raise FileNotFoundError(f'Missing {STUDY_ZIP}')
if MASK_DIR is None:
    raise FileNotFoundError(
        'Could not find LAD.nii.gz, LCX.nii.gz, and RCA.nii.gz in expected folders.'
    )

FALLBACK_SERIES = {'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
VESSELS = ['LAD', 'RCA', 'LCX']

REFERENCE_SPACING_SERIES_NUMBER = 7
MANUAL_REFERENCE_SPACING_XYZ_MM = (0.3515625, 0.3515625, 0.3)
MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3 = 0.5

LEFT_MAIN_CLINICAL_CALCIUM_VOLUME_MM3 = 54.0

print('Study zip:', STUDY_ZIP)
print('Mask directory:', MASK_DIR)


In [ ]:
def spacing_from_dicom_files(files, fallback_spacing):
    if not files:
        return tuple(float(x) for x in fallback_spacing), 'sitk_fallback_no_files'

    datasets = []
    for path in files:
        try:
            datasets.append(pydicom.dcmread(path, stop_before_pixels=True, force=True))
        except Exception:
            pass

    if not datasets:
        return tuple(float(x) for x in fallback_spacing), 'sitk_fallback_no_readable_dicom'

    first = datasets[0]
    try:
        row_spacing, col_spacing = [float(x) for x in first.PixelSpacing]
    except Exception:
        row_spacing, col_spacing = float(fallback_spacing[1]), float(fallback_spacing[0])

    positions = []
    for ds in datasets:
        ipp = getattr(ds, 'ImagePositionPatient', None)
        if ipp is not None and len(ipp) == 3:
            positions.append(np.asarray([float(x) for x in ipp], dtype=float))

    z_spacing = None
    if len(positions) > 1:
        positions = sorted(positions, key=lambda p: tuple(p.tolist()))
        deltas = [
            float(np.linalg.norm(positions[i + 1] - positions[i]))
            for i in range(len(positions) - 1)
        ]
        deltas = [d for d in deltas if d > 1e-6]
        if deltas:
            z_spacing = float(np.median(deltas))

    if z_spacing is None:
        for attr in ('SpacingBetweenSlices', 'SliceThickness'):
            value = getattr(first, attr, None)
            if value is not None:
                try:
                    z_spacing = float(value)
                    break
                except Exception:
                    pass

    if z_spacing is None:
        z_spacing = float(fallback_spacing[2])

    return (
        float(col_spacing),
        float(row_spacing),
        float(z_spacing),
    ), 'dicom_metadata'


def dicom_files_for_series_number(study, series_number):
    matches = [
        s for s in study.series
        if int(s.get('series_number', -1)) == int(series_number)
    ]
    if not matches:
        return []

    folder = matches[0]['folder']
    out = []
    for root, _, names in os.walk(folder):
        out.extend(os.path.join(root, n) for n in names)
    return out


study = OpenPlaqueStudy(str(STUDY_ZIP))
series_map = auto_detect_or_fallback_series(study, fallback_series=FALLBACK_SERIES)
series_map = {k: int(series_map[k]) for k in VESSELS}
print('Series map:', series_map)

ref_files = dicom_files_for_series_number(study, REFERENCE_SPACING_SERIES_NUMBER)
REFERENCE_SPACING, ref_source = spacing_from_dicom_files(
    ref_files, MANUAL_REFERENCE_SPACING_XYZ_MM
)

if float(np.prod(REFERENCE_SPACING)) >= MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3:
    REFERENCE_SPACING = MANUAL_REFERENCE_SPACING_XYZ_MM
    ref_source = 'manual_reference_spacing'

print('Reference spacing:', REFERENCE_SPACING, ref_source)


In [ ]:
volumes = {}
masks = {}
spacings = {}
images = {}
spacing_sources = {}

for artery in VESSELS:
    image, volume, dicom_files = study.load_series(series_map[artery])
    mask_img = sitk.ReadImage(str(MASK_DIR / f'{artery}.nii.gz'))
    mask = sitk.GetArrayFromImage(mask_img)

    dicom_spacing, spacing_source = spacing_from_dicom_files(
        dicom_files, image.GetSpacing()
    )
    analysis_spacing = dicom_spacing

    if float(np.prod(dicom_spacing)) >= MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3:
        analysis_spacing = REFERENCE_SPACING
        spacing_source = f'reference_series_{REFERENCE_SPACING_SERIES_NUMBER}'

    if volume.shape != mask.shape:
        raise ValueError(
            f'{artery}: DICOM volume shape {volume.shape} does not match '
            f'mask shape {mask.shape}.'
        )

    volumes[artery] = np.asarray(volume)
    masks[artery] = np.asarray(mask)
    spacings[artery] = tuple(float(x) for x in analysis_spacing)
    images[artery] = image
    spacing_sources[artery] = spacing_source

    print(
        f'{artery}: series={series_map[artery]}, shape={volume.shape}, '
        f'analysis spacing={analysis_spacing} ({spacing_source}), '
        f'labels={np.unique(mask)}'
    )


## Reproduce the earlier TPV calculations

The reusable `openplaque.plaque_volume` module implements the same HU categories and
one-voxel vessel-context expansion as the earlier best-estimate plaque-type notebook.


In [ ]:
plaque_estimates = {}

for artery in VESSELS:
    plaque_estimates[artery] = estimate_best_estimate_plaque_volume(
        artery=artery,
        volume=volumes[artery],
        segmentation_mask=masks[artery],
        spacing_xyz_mm=spacings[artery],
        vessel_dilation_voxels=1,
        exclude_vessel_label_for_candidates=True,
        low_attenuation_min_hu=-30,
        low_attenuation_max_hu=30,
        fibrofatty_min_hu=30,
        fibrofatty_max_hu=130,
        fibrous_min_hu=130,
        fibrous_max_hu=350,
        dense_calcium_min_hu=350,
    )

tpv_df = pd.DataFrame([plaque_estimates[a].to_row() for a in VESSELS])

display(
    tpv_df[
        [
            'artery',
            'strict_nnunet_core_volume_mm3',
            'context_expanded_candidate_volume_mm3',
            'low_attenuation_mm3',
            'fibrofatty_mm3',
            'fibrous_mm3',
            'dense_calcium_mm3',
            'total_plaque_volume_proxy_mm3',
        ]
    ].round(3)
)


In [ ]:
strict_three_vessel_tpv = float(tpv_df['strict_nnunet_core_volume_mm3'].sum())
best_three_vessel_tpv = float(tpv_df['total_plaque_volume_proxy_mm3'].sum())
best_plus_left_main_tpv = best_three_vessel_tpv + LEFT_MAIN_CLINICAL_CALCIUM_VOLUME_MM3

earlier_tpv_comparison = pd.DataFrame([
    {
        'definition': 'Earlier strict TPV: LAD + RCA + LCX',
        'tpv_mm3': strict_three_vessel_tpv,
        'left_main_included': False,
    },
    {
        'definition': 'Earlier best-estimate TPV proxy: LAD + RCA + LCX',
        'tpv_mm3': best_three_vessel_tpv,
        'left_main_included': False,
    },
    {
        'definition': 'Earlier best-estimate TPV proxy + clinical left main',
        'tpv_mm3': best_plus_left_main_tpv,
        'left_main_included': True,
    },
])

display(earlier_tpv_comparison.round(3))
print('Clinical left-main add-on:', LEFT_MAIN_CLINICAL_CALCIUM_VOLUME_MM3, 'mm³')


## Estimate one candidate outer wall per artery

For each artery, the outer-wall seed includes the **best-estimate plaque proxy**, so the
candidate outer wall necessarily contains both the strict core and the broader plaque proxy.
Both PAV definitions below therefore use the same denominator for a given artery.


In [ ]:
MAX_WALL_THICKNESS_MM = 2.0
FAT_THRESHOLD_HU = -30.0

outer_masks = {}
pav_rows = []

for artery in VESSELS:
    pe = plaque_estimates[artery]

    seed_mask = segmentation_mask_with_plaque_proxy(
        masks[artery],
        pe.best_estimate_mask,
    )

    outer = estimate_outer_wall_candidate(
        volume=volumes[artery],
        mask=seed_mask,
        spacing=spacings[artery],
        max_wall_thickness_mm=MAX_WALL_THICKNESS_MM,
        fat_threshold_hu=FAT_THRESHOLD_HU,
    )
    outer_masks[artery] = outer

    strict_vol, outer_vol, strict_pav, _, _ = compute_pav(
        pe.strict_core_mask, outer, spacings[artery]
    )
    best_vol, _, best_pav, _, _ = compute_pav(
        pe.best_estimate_mask, outer, spacings[artery]
    )

    pav_rows.append({
        'artery': artery,
        'strict_tpv_mm3': strict_vol,
        'best_estimate_tpv_proxy_mm3': best_vol,
        'candidate_outer_vessel_volume_mm3': outer_vol,
        'strict_core_pav_percent': strict_pav,
        'best_estimate_pav_percent': best_pav,
    })

pav_df = pd.DataFrame(pav_rows)
display(pav_df.round(3))


In [ ]:
total_outer = float(pav_df['candidate_outer_vessel_volume_mm3'].sum())
strict_pav_total = 100.0 * strict_three_vessel_tpv / total_outer if total_outer else 0.0
best_pav_total = 100.0 * best_three_vessel_tpv / total_outer if total_outer else 0.0

whole_heart_pav_df = pd.DataFrame([
    {
        'definition': '3-vessel strict-core experimental PAV',
        'plaque_volume_mm3': strict_three_vessel_tpv,
        'outer_vessel_volume_mm3': total_outer,
        'pav_percent': strict_pav_total,
    },
    {
        'definition': '3-vessel best-estimate experimental PAV',
        'plaque_volume_mm3': best_three_vessel_tpv,
        'outer_vessel_volume_mm3': total_outer,
        'pav_percent': best_pav_total,
    },
])

display(whole_heart_pav_df.round(3))

print()
print('Left-main 54 mm³ is intentionally excluded from PAV because')
print('this prototype has no corresponding left-main outer-vessel denominator.')


## Visual QC

Yellow/solid contours show the candidate outer wall. Dashed contours show the strict
nnU-Net plaque core. The additional figure shades the broader best-estimate plaque proxy.


In [ ]:
for artery in VESSELS:
    pe = plaque_estimates[artery]
    counts = np.sum(pe.best_estimate_mask, axis=(1, 2))
    z = int(np.argmax(counts))

    print(f'{artery}: z={z}')

    # Existing strict-core + outer-wall overlay
    show_pav_overlay(
        volumes[artery],
        masks[artery],
        outer_masks[artery],
        z=z,
    )
    plt.show()

    # Best-estimate proxy overlay
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(volumes[artery][z], cmap='gray', vmin=-200, vmax=800)
    ax.imshow(
        np.ma.masked_where(~pe.best_estimate_mask[z], pe.best_estimate_mask[z]),
        alpha=0.45,
    )
    ax.set_title(f'{artery}: best-estimate plaque proxy, slice {z}')
    ax.axis('off')
    plt.show()


## Save CSVs and candidate outer-wall masks to Drive


In [ ]:
OUT_DIR = DRIVE_ROOT / 'PAV_Best_Estimate_Standalone'
OUT_DIR.mkdir(parents=True, exist_ok=True)

tpv_df.to_csv(OUT_DIR / 'tpv_by_artery.csv', index=False)
earlier_tpv_comparison.to_csv(OUT_DIR / 'earlier_tpv_comparison.csv', index=False)
pav_df.to_csv(OUT_DIR / 'pav_by_artery.csv', index=False)
whole_heart_pav_df.to_csv(OUT_DIR / 'whole_heart_pav_summary.csv', index=False)

for artery in VESSELS:
    pe = plaque_estimates[artery]

    outer_img = sitk.GetImageFromArray(outer_masks[artery].astype(np.uint8))
    outer_img.CopyInformation(images[artery])
    sitk.WriteImage(
        outer_img,
        str(OUT_DIR / f'{artery}_candidate_outer_wall.nii.gz')
    )

    proxy_img = sitk.GetImageFromArray(pe.best_estimate_mask.astype(np.uint8))
    proxy_img.CopyInformation(images[artery])
    sitk.WriteImage(
        proxy_img,
        str(OUT_DIR / f'{artery}_best_estimate_plaque_proxy.nii.gz')
    )

print('Saved to:', OUT_DIR)


## Optional: run the repository unit tests


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q',
     str(REPO_DIR / 'tests/test_plaque_volume.py'),
     str(REPO_DIR / 'tests/test_pav.py')],
    check=True,
)
